# 12. (2 points)

The Exponential Dispersion Family ("EDF") is a family of distributions with probability density function $\pi(y;\theta, \phi)$ of the form:

$$
\mathrm{ln}\pi(y;\theta,\phi) = \frac{y\theta-b(\theta)}{a(\phi)}+c(y,\phi)
$$

and variance function of the form:

$$
V(\mu) = b''((b')^{-1}(\mu))
$$

* a.) (0.5 point) Briefly describe the parameter $\theta$ and the function $b(\theta)$ in the equation above.
* b.) (0.5 point) Explain how the Tweedie sub-family can be obtained from the EDF above.
* c.) (0.25 point) Briefly describe how to select a Tweedie distribution for a heavy-tailed exposure.

Given the following information as of December 31, 2018:

In [1]:
import chainladder as cl

tri = cl.Triangle(
    data={
        "Accident Year": [2016, 2016, 2016, 2017, 2017, 2018],
        "Development": [2016, 2017, 2018, 2017, 2018, 2018],
        "Paid Loss": [180, 1000, 1200, 200, 600, 150],
    },
    origin="Accident Year",
    development="Development",
    columns="Paid Loss",
    cumulative=True
)

print(tri)

         12      24      36
2016  180.0  1000.0  1200.0
2017  200.0   600.0     NaN
2018  150.0     NaN     NaN


# A GLM is used to model the incremental paid losses. The GLM uses a log link function and a Tweedie error distribution specified by:

- $p=1$
- $b(\theta)=\mathrm{exp}(\theta)$
- $\mu=\mathrm{exp}(\theta)$
- $\mathrm{ln}\pi(y;\mu,\phi) = [y \mathrm{ln} \mu - \mu] / \phi$

* d. (0.75 point) Calculate the accident year 2018 ultimate loss.

## Solution (Taylor, *Stochastic Loss Reserving Using GLMs*, Ch. 2-3)

**a.** $\theta$ is the **canonical (location) parameter** of the EDF. It is linked to the mean by $E[Y]=b'(\theta)$, i.e. $\theta=(b')^{-1}(\mu)$. $b(\cdot)$ is the **cumulant function**. It determines the shape of the distribution: $E[Y]=b'(\theta)$ and $Var[Y]=a(\phi)\,b''(\theta)$, and so the variance function $V(\mu)=b''((b')^{-1}(\mu))$. Choosing $b$ (with $a$, $c$) selects the distribution, e.g. $b(\theta)=e^\theta$ gives the Poisson, $b(\theta)=\theta^2/2$ the Normal.

**b.** The Tweedie sub-family is obtained by restricting the variance function to a power of the mean, $V(\mu)=\mu^p$ with $p\le0$ or $p\ge1$, so that $Var[Y]=\phi\,\mu^p$. This forces the cumulant function to be $b(\theta)=\frac{1}{2-p}[(1-p)\theta]^{(2-p)/(1-p)}$ (and $\mu=[(1-p)\theta]^{1/(1-p)}$). Special cases: $p=0$ Normal, $p=1$ (over-dispersed) Poisson, $p=2$ Gamma, $p=3$ Inverse Gaussian.

**c.** Tail heaviness increases with $p$. So for a heavy-tailed exposure choose a Tweedie member with a larger power $p$ (e.g. Gamma, $p=2$, or higher rather than Poisson, $p=1$); in practice, if a model with index $p$ gives residuals more widely dispersed than the model implies, consider increasing $p$.

**d.** With $p=1$ and a log link on incremental paid losses, the covariates are accident year and development period (the cross-classified model). Taylor Section 3.3 shows that the ODP cross-classified GLM reproduces the chain ladder (volume-weighted) forecast. The problem does not state the covariates, so I assume this standard structure. I fit the GLM directly and check it against the chain ladder.

In [2]:
import numpy as np
import pandas as pd
import patsy
from sklearn.linear_model import TweedieRegressor

# incremental paid, long format
inc = tri.cum_to_incr().to_frame()
long = (inc.stack().dropna().rename("y").reset_index()
        .rename(columns={"level_0": "origin", "level_1": "dev"}))
long["origin"] = long["origin"].astype(str).str[:4]
long["dev"] = long["dev"].astype(int)

X = patsy.dmatrix("C(origin) + C(dev)", long)
glm = TweedieRegressor(power=1, link="log", alpha=0, fit_intercept=False, max_iter=10000, tol=1e-12)
glm.fit(np.asarray(X), long["y"])

future = pd.DataFrame({"origin": ["2018", "2018", "2017"], "dev": [24, 36, 36]})
mu = glm.predict(np.asarray(patsy.build_design_matrices([X.design_info], future)[0]))
future["mu"] = mu
print(future)

ult_2018 = 150 + mu[0] + mu[1]
print("AY2018 ultimate (GLM):", round(ult_2018, 2))
ult_cl = cl.Chainladder().fit(tri).ultimate_.to_frame().iloc[2, 0]
print("AY2018 ultimate (chain ladder):", round(ult_cl, 2))
assert np.isclose(ult_2018, ult_cl, atol=1e-3) and np.isclose(ult_2018, 757.89, atol=0.01)

  origin  dev          mu
0   2018   24  481.578951
1   2018   36  126.315795
2   2017   36  120.000004
AY2018 ultimate (GLM): 757.89
AY2018 ultimate (chain ladder): 757.89


**d.** AY 2018 ultimate $=150+481.6+126.3=$ **≈ 757.9**, identical to the chain ladder: $150\times\frac{1600}{380}\times\frac{1200}{1000}=757.9$.

## chainladder-python gaps

- **Bug in `cl.TweedieGLM`:** the `alpha` parameter is documented ("`alpha = 0` is equivalent to unpenalized GLMs") but is **never passed** to the underlying `sklearn.TweedieRegressor`, so the default ridge penalty (`alpha=1`) is always applied. The penalized fit does not give the MLE: for this triangle `TweedieGLM(power=1, alpha=0)` gives `ldf_` = 4.4395, 1.2083 (the same as `alpha=1`) instead of the chain ladder's 4.2105, 1.2, and its fitted values don't reproduce the row and column totals. So the GLM answer here would be wrong (AY2018 ultimate 804.7 vs 757.9). I confirmed by calling sklearn directly with `alpha=0`.
- `TweedieGLM` has no `ultimate_`/`ibnr_`, no fitted-incremental forecast for the future cells, no `scale_` (dispersion $\phi$), no deviance or Pearson residuals, so it can't give prediction errors (Taylor Chapters 4-5).
- Suggested feature: pass `alpha` through, and add `ultimate_`, `ibnr_`, `scale_` and `residuals_`; see the sketch.

## Sketch: `TweedieGLM` with `alpha` passed through, plus reserve outputs

In [3]:
from sklearn.base import BaseEstimator


class ODPGLM(BaseEstimator):
    """Cross-classified Tweedie GLM on incremental losses (Taylor ch. 3.3)."""

    def __init__(self, power=1.0, link="log", alpha=0.0, design_matrix="C(origin) + C(dev)"):
        self.power, self.link, self.alpha, self.design_matrix = power, link, alpha, design_matrix

    def fit(self, X, y=None):
        inc = X.cum_to_incr().to_frame()
        d = (inc.stack().dropna().rename("y").reset_index().rename(columns={"level_0": "origin", "level_1": "dev"}))
        d["origin"] = d["origin"].astype(str).str[:4]; d["dev"] = d["dev"].astype(int)
        self.design_ = patsy.dmatrix(self.design_matrix, d)
        self.model_ = TweedieRegressor(power=self.power, link=self.link, alpha=self.alpha,
                                       fit_intercept=False, max_iter=10000, tol=1e-12)
        self.model_.fit(np.asarray(self.design_), d["y"])
        d["mu"] = self.model_.predict(np.asarray(self.design_))
        self.fitted_ = d
        # Pearson dispersion: sum (y-mu)^2 / mu^p / (n - p_params)
        n, k = len(d), np.asarray(self.design_).shape[1]
        self.scale_ = ((d["y"] - d["mu"]) ** 2 / d["mu"] ** self.power).sum() / (n - k)
        # forecast the lower triangle
        origins = sorted(d["origin"].unique()); devs = sorted(d["dev"].unique())
        seen = set(zip(d["origin"], d["dev"]))
        fut = pd.DataFrame([(o, v) for o in origins for v in devs if (o, v) not in seen], columns=["origin", "dev"])
        Xf = patsy.build_design_matrices([self.design_.design_info], fut)[0]
        fut["mu"] = self.model_.predict(np.asarray(Xf))
        self.forecast_ = fut
        latest = d.groupby("origin")["y"].sum()
        self.ibnr_ = fut.groupby("origin")["mu"].sum().reindex(latest.index).fillna(0.0)
        self.ultimate_ = latest + self.ibnr_
        return self


est = ODPGLM(power=1, alpha=0).fit(tri)
print(est.ultimate_.round(2).to_dict(), " scale_:", round(est.scale_, 3))
assert np.isclose(est.ultimate_["2018"], ult_2018)

{'2016': 1200.0, '2017': 720.0, '2018': 757.89}  scale_: 48.686


Extensions not prototyped: `residuals_` (deviance/Pearson), prediction error (Taylor ch. 4), other covariate structures (e.g. calendar-year or parametric development), and a bootstrap wrapper (Taylor ch. 5).